# 2.5 — Random Forest + export (Forma 2, features do raw)

**Mesmo modelo da Forma 1** (Random Forest + StandardScaler + export **micromlgen**), agora
sobre as features que nós mesmos extraímos do raw no notebook **2.4**
(`features_from_raw.csv`). O split treino/teste já vem decidido (coluna `split`).

> **Atende Sprint 4:** item 2 (treino/teste + métricas + matriz de confusão) e item 3
> (feature importance + interpretação). A comparação com a Forma 1 (notebook 1.5) é o ponto
> didático: **mesmo problema, mesmo modelo, caminhos de dados diferentes**.

In [ ]:
!pip install -q pandas scikit-learn matplotlib micromlgen

## 1) Carregar as features (saída do 2.4)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    enviados = files.upload()
    nome = list(enviados.keys())[0]
except Exception:
    nome = "features_from_raw.csv"

df = pd.read_csv(nome)
print(df.groupby(["label", "split"]).size())
df.head()

## 2) X/y e split por coluna `split` (sem vazamento)

As 7 features; classes `normal` (0) × `anomalo` (1). O split cronológico por classe já foi
feito no 2.4 — aqui só respeitamos a coluna `split`.

In [ ]:
FEATURES = ["mean_ax", "mean_ay", "mean_az", "std_ax", "std_ay", "std_az", "rms_mag"]
CLASSES  = {"normal": 0, "anomalo": 1}

df = df[df["label"].isin(CLASSES)].copy()
df["y"] = df["label"].map(CLASSES)

tr = df[df["split"] == "treino"]
te = df[df["split"] == "teste"]
X_train, y_train = tr[FEATURES].values, tr["y"].values
X_test,  y_test  = te[FEATURES].values, te["y"].values
print(f"Treino: {len(y_train)} | Teste: {len(y_test)}")

## 3) StandardScaler + Random Forest (mesma config da Forma 1)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=42)
clf.fit(X_train_s, y_train)

## 4) Métricas e matriz de confusão

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

y_pred = clf.predict(X_test_s)
print("Acurácia:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, target_names=["normal", "anomalo"]))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["normal", "anomalo"]).plot()
plt.show()

## 5) Feature importance + interpretação física

In [ ]:
from sklearn.inspection import permutation_importance

imp_nativa = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
perm = permutation_importance(clf, X_test_s, y_test, n_repeats=20, random_state=42)
imp_perm = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp_nativa.plot.barh(ax=axes[0], color="tab:blue");  axes[0].set_title("Importância nativa (RF)")
imp_perm.plot.barh(ax=axes[1], color="tab:orange");  axes[1].set_title("Permutation importance (teste)")
plt.tight_layout(); plt.show()

**Interpretação:** como na Forma 1, espera-se `rms_mag` e `std_*` no topo — vibração eleva a
intensidade efetiva e a variabilidade, enquanto a média quase não muda (Aula 14, slides 12–13).

## 6) Exportar para o ESP32 (micromlgen)

Gera os mesmos dois headers da Forma 1. O firmware embarcado fica para depois.

In [ ]:
from micromlgen import port

with open("AIoTVibracaoRF_micromlgen.hpp", "w") as f:
    f.write(port(clf))

def gerar_scaler_hpp(scaler, n):
    means  = ", ".join(f"{m:.10f}f" for m in scaler.mean_)
    scales = ", ".join(f"{s:.10f}f" for s in scaler.scale_)
    return f'''#ifndef STANDARD_SCALER_HPP
#define STANDARD_SCALER_HPP
// StandardScaler das 7 features de vibracao: mean_ax/ay/az, std_ax/ay/az, rms_mag
namespace Scaler {{
    const static float means[{n}]  = {{ {means} }};
    const static float scales[{n}] = {{ {scales} }};
    inline void std(const float* input, float* output) {{
        for (int i = 0; i < {n}; i++) output[i] = (input[i] - means[i]) / scales[i];
    }}
}}
#endif
'''

with open("AIoTVibracaoScaler.hpp", "w") as f:
    f.write(gerar_scaler_hpp(scaler, len(FEATURES)))

try:
    from google.colab import files
    files.download("AIoTVibracaoRF_micromlgen.hpp")
    files.download("AIoTVibracaoScaler.hpp")
except Exception:
    print("Arquivos gerados na pasta atual (fora do Colab).")